# 2. Tính toán chỉ số kiểm thử Khử trùng lặp cùng nguồn & đa nguồn
Notebook này phân loại và tính toán chi tiết các chỉ số **Precision, Recall, F1-Score** cho 2 kịch bản:
1. **Cùng nguồn (Same-Source)**
2. **Đa nguồn / Chéo nguồn (Cross-Source)**


## Bước 2.1: Nạp dữ liệu và Phân nhóm kịch bản


In [6]:
import json
from pathlib import Path

workspace_root = Path(r'f:\HCMUS_KH\LuanVan\JobVisualization_BE')
script_dir = workspace_root / 'KiemThu' / 'KiemThu_Deduplication'

gt_path = script_dir / 'ground_truth_deduplication.json'
with open(gt_path, 'r', encoding='utf-8') as f:
    pairs = json.load(f)

same_source_pairs = [p for p in pairs if p['scenario'] == 'SAME_SOURCE']
cross_source_pairs = [p for p in pairs if p['scenario'] == 'CROSS_SOURCE']

print(f'Nạp thành công {len(pairs)} cặp:')
print(f'  - Cùng nguồn (Same-Source): {len(same_source_pairs)} cặp')
print(f'  - Đa nguồn (Cross-Source) : {len(cross_source_pairs)} cặp')


Nạp thành công 52 cặp:
  - Cùng nguồn (Same-Source): 17 cặp
  - Đa nguồn (Cross-Source) : 35 cặp


## Bước 2.2: Hàm tính toán Precision, Recall, F1-Score


In [7]:
def evaluate_scenario(scenario_pairs, name):
    TP = FP = TN = FN = 0
    for p in scenario_pairs:
        pred = p['predicted_label']
        true = p['true_label']
        if pred == 'DUPLICATE' and true == 'DUPLICATE':
            TP += 1
        elif pred == 'DUPLICATE' and true == 'INDEPENDENT':
            FP += 1
        elif pred == 'INDEPENDENT' and true == 'DUPLICATE':
            FN += 1
        else:
            TN += 1
            
    precision = TP / (TP + FP) if (TP + FP) > 0 else 0.0
    recall = TP / (TP + FN) if (TP + FN) > 0 else 0.0
    f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0.0
    
    print(f'=== KẾT QUẢ ĐÁNH GIÁ: {name} ===')
    print(f'  Confusion Matrix: TP={TP}, FP={FP}, TN={TN}, FN={FN}')
    print(f'  Precision (Độ chính xác) : {precision*100:.2f}% ({TP}/{TP+FP})')
    print(f'  Recall (Độ phủ)          : {recall*100:.2f}% ({TP}/{TP+FN})')
    print(f'  F1-Score (Chỉ số F1)     : {f1*100:.2f}%')
    print('='*50)
    return {'TP': TP, 'FP': FP, 'TN': TN, 'FN': FN, 'precision': precision, 'recall': recall, 'f1': f1}

same_metrics = evaluate_scenario(same_source_pairs, 'CÙNG NGUỒN (SAME-SOURCE)')
cross_metrics = evaluate_scenario(cross_source_pairs, 'ĐA NGUỒN (CROSS-SOURCE)')
overall_metrics = evaluate_scenario(pairs, 'TỔNG THỂ CHUNG CUỘC (OVERALL)')


=== KẾT QUẢ ĐÁNH GIÁ: CÙNG NGUỒN (SAME-SOURCE) ===
  Confusion Matrix: TP=0, FP=0, TN=17, FN=0
  Precision (Độ chính xác) : 0.00% (0/0)
  Recall (Độ phủ)          : 0.00% (0/0)
  F1-Score (Chỉ số F1)     : 0.00%
=== KẾT QUẢ ĐÁNH GIÁ: ĐA NGUỒN (CROSS-SOURCE) ===
  Confusion Matrix: TP=4, FP=0, TN=31, FN=0
  Precision (Độ chính xác) : 100.00% (4/4)
  Recall (Độ phủ)          : 100.00% (4/4)
  F1-Score (Chỉ số F1)     : 100.00%
=== KẾT QUẢ ĐÁNH GIÁ: TỔNG THỂ CHUNG CUỘC (OVERALL) ===
  Confusion Matrix: TP=4, FP=0, TN=48, FN=0
  Precision (Độ chính xác) : 100.00% (4/4)
  Recall (Độ phủ)          : 100.00% (4/4)
  F1-Score (Chỉ số F1)     : 100.00%


## Bước 2.3: In bảng đối soát chi tiết (Markdown cho Luận văn)


In [8]:
print('### Bảng đối soát các cặp tin trùng lặp mẫu cùng nguồn (Same-Source):')
print('| STT | Công ty | Nguồn | Tiêu đề A | Tiêu đề B | Sim | Dự đoán | Ground Truth | Trạng thái |')
print('|---|---|---|---|---|---|---|---|---|')
idx = 1
for p in same_source_pairs:
    pred = p['predicted_label']
    true = p['true_label']
    if pred == 'DUPLICATE' and true == 'DUPLICATE': status = 'TP'
    elif pred == 'DUPLICATE' and true == 'INDEPENDENT': status = 'FP (Gộp nhầm)'
    elif pred == 'INDEPENDENT' and true == 'DUPLICATE': status = 'FN (Bỏ sót)'
    else: continue # Chi show cac truong hop trung lap thuc te/du doan de bang ngan gon
    
    ta = (p['title_a'] or 'N/A')[:25]
    tb = (p['title_b'] or 'N/A')[:25]
    comp = (p['company_name'] or 'N/A')[:20]
    print(f'| {idx} | {comp} | {p["source_a"]} | {ta} | {tb} | {p["similarity"]} | {pred} | {true} | {status} |')
    idx += 1

print("\n")
print('### Bảng đối soát các cặp tin trùng lặp mẫu đa nguồn (Cross-Source):')
print('| STT | Công ty | Nguồn A | Nguồn B | Tiêu đề A | Tiêu đề B | Sim | Dự đoán | Ground Truth | Trạng thái |')
print('|---|---|---|---|---|---|---|---|---|---|')
idx = 1
for p in cross_source_pairs:
    pred = p['predicted_label']
    true = p['true_label']
    if pred == 'DUPLICATE' and true == 'DUPLICATE': status = 'TP'
    elif pred == 'DUPLICATE' and true == 'INDEPENDENT': status = 'FP (Gộp nhầm)'
    elif pred == 'INDEPENDENT' and true == 'DUPLICATE': status = 'FN (Bỏ sót)'
    else: continue
    
    ta = (p['title_a'] or 'N/A')[:25]
    tb = (p['title_b'] or 'N/A')[:25]
    comp = (p['company_name'] or 'N/A')[:20]
    print(f'| {idx} | {comp} | {p["source_a"]} | {p["source_b"]} | {ta} | {tb} | {p["similarity"]} | {pred} | {true} | {status} |')
    idx += 1


### Bảng đối soát các cặp tin trùng lặp mẫu cùng nguồn (Same-Source):
| STT | Công ty | Nguồn | Tiêu đề A | Tiêu đề B | Sim | Dự đoán | Ground Truth | Trạng thái |
|---|---|---|---|---|---|---|---|---|


### Bảng đối soát các cặp tin trùng lặp mẫu đa nguồn (Cross-Source):
| STT | Công ty | Nguồn A | Nguồn B | Tiêu đề A | Tiêu đề B | Sim | Dự đoán | Ground Truth | Trạng thái |
|---|---|---|---|---|---|---|---|---|---|
| 1 | FPT Software | itviec | vietnamworks | Project Manager (JP N2+/  | Project Manager (JP N2+/  | 1.0 | DUPLICATE | DUPLICATE | TP |
| 2 | FPT Software | itviec | vietnamworks | .NET Developer (English) | .NET Developer (English)  | 1.0 | DUPLICATE | DUPLICATE | TP |
| 3 | FPT Software | vietnamworks | itviec | Technical Advisor | Technical Advisor (Cross- | 0.9913 | DUPLICATE | DUPLICATE | TP |
| 4 | Công Ty Cổ Phần Sữa  | itviec | vietnamworks | Associate IT Manager (Pro | Associate IT Manager - Co | 0.9684 | DUPLICATE | DUPLICATE | TP |
